# MonoDGP M57: complete portable-attention validation

Run every cell on a CUDA GPU. This continuation reuses the exact reviewed M57 manifest and smoke report; it does not regenerate them. It runs no training and no Core ML conversion. Its only purpose is to evaluate the portable path on all 3,769 Chen-validation images against the frozen M56d preservation gates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import hashlib, json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M57')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m56d')
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m57_deformable_attention')
MANIFEST=OUTPUT_ROOT/'m57_deformable_attention_manifest.json'
SMOKE=OUTPUT_ROOT/'m57_deformable_attention_smoke.json'
COMPLETE_OUTPUT=OUTPUT_ROOT/'complete_evaluation'
LOG_DIR=OUTPUT_ROOT/'colab_logs'
REVIEWED={
    MANIFEST:'7c4757798dfff4d95053912730faff4c5d7a4626ff41a539ad87b2f9193eb313',
    SMOKE:'2a96cffd8e1e6b77f2c547c2b94dca4bde2e72bf4185d853ee7bca71ed28b3b0',
}
def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
def run(command,cwd=None,env=None):
    command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_logged(command,cwd,log_path,env=None):
    command=[str(item) for item in command]; print('+',shlex.join(command),flush=True)
    log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True)
    merged=os.environ.copy(); merged.update(env or {}); tail=deque(maxlen=100)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=cwd,env=merged,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=process.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
    return log_path
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run(['nvidia-smi'])

In [ ]:
# Restore exact source and operator patches in a fresh Colab runtime.
if not MOBILE_REPO.exists():
    run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else:
    run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists():
    run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m54_training.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m57_deformable_attention.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
changed=set(subprocess.run(['git','diff','--name-only'],cwd=MONODGP_REPO,check=True,capture_output=True,text=True).stdout.splitlines())
expected={
    'lib/datasets/kitti/kitti_dataset.py','lib/helpers/save_helper.py',
    'lib/helpers/trainer_helper.py','lib/models/monodgp/ops/modules/ms_deform_attn.py',
    'lib/models/monodgp/ops/setup.py','lib/models/monodgp/ops/src/cuda/ms_deform_attn_cuda.cu',
    'tools/train_val.py',
}
if changed != expected: raise RuntimeError(f'Unexpected patched source set: {changed}')
ops=MONODGP_REPO/'lib/models/monodgp/ops'
shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))'],cwd=MONODGP_REPO)

In [ ]:
# Restore the exact dataset view and verify reviewed evidence without rerunning smoke.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={
    key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
    for key,names in {
        'image_2':['training/image_2','training/image_02'],
        'label_2':['training/label_2','training/label_02'],
        'calib':['training/calib'],
    }.items()
}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True)
(DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'):
    shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
for path,expected_hash in REVIEWED.items():
    if not path.is_file(): raise FileNotFoundError(path)
    actual=sha256(path)
    if actual != expected_hash: raise RuntimeError(f'Reviewed evidence changed: {path}: {actual}')
print('Exact reviewed M57 evidence found; complete validation is authorized.')

In [ ]:
# Run portable inference on 3,769 images and all frozen accuracy/recall/localization gates.
EVAL_LOG=run_logged([
    sys.executable,'-u','scripts/evaluate_monodgp_m57_deformable_attention.py',
    '--mobile-repo',MOBILE_REPO,'--monodgp-repo',MONODGP_REPO,
    '--manifest',MANIFEST,'--smoke',SMOKE,
    '--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,
    '--output-dir',COMPLETE_OUTPUT,
    '--product-config','configs/kitti_mobileadas3d_s1.yaml','--profile','colab_drive',
],MOBILE_REPO,LOG_DIR/'m57_complete_evaluation.log')
GATE=COMPLETE_OUTPUT/'m57_portable_attention_gate.json'
COMPARISON=COMPLETE_OUTPUT/'m57_portable_attention_comparison.csv'
gate=json.loads(GATE.read_text())
print('Portable metrics:',json.dumps(gate['portable_metrics'],indent=2))
print('Preservation gates:',json.dumps(gate['preservation_gate_results'],indent=2))
print('Reviewed native/portable runtime:',json.dumps(gate['runtime_comparison'],indent=2))
assert gate['portable_operator_candidate_selected']
assert gate['next_coreml_conversion_gate_authorized']
assert gate['direct_coreml_conversion_authorized'] is False
assert gate['deployment_authorized'] is False

## Stop point 2

Return m57_portable_attention_gate.json and m57_portable_attention_comparison.csv from the complete_evaluation directory. Passing this notebook selects the portable operator candidate and authorizes a separate Core ML conversion experiment; it does not itself authorize conversion, deployment, or product safety.